# Lecture 1: Matrix-Vector Multiplication
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Matrix Times Vector

$Ax$ is a **linear combination of the columns of $A$**, with coefficients given by the entries of $x$.

In [ ]:
# Define a magic square matrix and a vector
A = np.array([[8, 1, 6],
              [3, 5, 7],
              [4, 9, 2]], dtype=float)
x = np.array([-1, 2, 1], dtype=float)

print("A =")
print(A)
print("\nx =", x)

In [ ]:
# Standard matrix-vector product
print("A @ x =", A @ x)

In [ ]:
# Same result as a linear combination of columns
result = x[0] * A[:, 0] + x[1] * A[:, 1] + x[2] * A[:, 2]
print("x[0]*A[:,0] + x[1]*A[:,1] + x[2]*A[:,2] =", result)

## Quasimatrix / Vandermonde-style Matrix

A polynomial $c_0 + c_1 t + \cdots + c_n t^n$ can be written as a matrix-vector product where the "matrix" has columns $1, t, t^2, \ldots$

We discretize $t \in [0,1]$ and build the matrix explicitly.

In [ ]:
t = np.linspace(0, 1, 51)
V = np.column_stack([t**j for j in range(4)])

plt.figure(figsize=(6, 4))
for j in range(4):
    plt.plot(t, V[:, j], label=f"$t^{j}$")
plt.xlabel("t")
plt.ylabel("$t^j$")
plt.title("Columns of the Vandermonde-style matrix")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Matrix Times Matrix

A matrix-matrix product $C = AB$ can be viewed as a collection of matrix-vector products: each column of $C$ is $A$ times the corresponding column of $B$.

In [ ]:
# magic(4) equivalent
A = np.array([[16,  2,  3, 13],
              [ 5, 11, 10,  8],
              [ 9,  7,  6, 12],
              [ 4, 14, 15,  1]], dtype=float)

# Upper triangular matrix of ones (4x3)
B = np.triu(np.ones((4, 3)))

C = A @ B
print("C = A @ B =")
print(C)

In [ ]:
# Verify: each column of C is A times the corresponding column of B
C_columnwise = np.column_stack([A @ B[:, j] for j in range(3)])
print("Column-by-column computation:")
print(C_columnwise)

## Outer Product

When $A$ is a column vector $u$ and $B$ is a row vector $v^*$, the product $uv^*$ is a **rank-1 outer product**.

In [ ]:
u = np.array([[1], [2], [3], [4]])
v = np.array([[1j, -1j, 1]])

print("u shape:", u.shape)
print("v shape:", v.shape)
print("\nOuter product u @ v:")
print(u @ v)

## Rank and Inverse

Rank and invertibility are **not robust to perturbations**.

In [ ]:
A = np.array([[0, 1],
              [0, 0]], dtype=float)

print("A =")
print(A)
print("rank(A) =", np.linalg.matrix_rank(A))

In [ ]:
# A tiny perturbation changes the rank
B = A + np.array([[0, 0],
                   [1e-12, 0]])

print("B =")
print(B)
print("rank(B) =", np.linalg.matrix_rank(B))

In numerical computation, representation of real numbers cannot be exact. The notions of rank and invertibility must be carefully reconsidered in finite precision.

## Change of Basis

If the columns of $A$ form a basis, then $v_{\text{standard}} = Ax$ converts from $A$-basis coordinates to standard coordinates, and $x = A^{-1} v$ converts back.

In [ ]:
A = np.array([[8, 1, 6],
              [3, 5, 7],
              [4, 9, 2]], dtype=float)
x = np.array([-1, 1, 2], dtype=float)

v_standard = A @ x
print("v in standard basis =", v_standard)

In [ ]:
# Convert back using the inverse (not recommended in practice)
v_Abasis_inv = np.linalg.inv(A) @ v_standard
print("v in A-basis (via inv) =", v_Abasis_inv)

In [ ]:
# Preferred: use np.linalg.solve (equivalent to MATLAB's backslash)
v_Abasis_solve = np.linalg.solve(A, v_standard)
print("v in A-basis (via solve) =", v_Abasis_solve)

## Two Views of $Ax$

### View 1: Row-oriented (dot products)
Each entry $b_i$ is the dot product of row $i$ of $A$ with $x$.

### View 2: Column-oriented (linear combination)
$Ax$ is a linear combination of the columns of $A$, weighted by entries of $x$.

In [ ]:
A = np.array([[1, 2],
              [3, 4]], dtype=float)
x = np.array([5, 6], dtype=float)

# View 1: dot products of rows with x
b_row = np.array([A[i] @ x for i in range(A.shape[0])])
print("Row view (dot products):  ", b_row)

# View 2: linear combination of columns
b_col = x[0] * A[:, 0] + x[1] * A[:, 1]
print("Column view (lin. comb.): ", b_col)

## Range, Nullspace, and Rank

Demonstrating with a rank-deficient matrix.

In [ ]:
A = np.array([[1, 2],
              [2, 4]], dtype=float)

print("A =")
print(A)
print("rank(A) =", np.linalg.matrix_rank(A))

In [ ]:
# The nullspace: Ax = 0  =>  x1 + 2*x2 = 0  =>  nullspace = span{[-2, 1]}
x_null = np.array([-2, 1], dtype=float)
print("A @ [-2, 1] =", A @ x_null, " (should be zero)")

# rank + nullspace dimension = number of columns
print(f"rank ({np.linalg.matrix_rank(A)}) + nullspace dim (1) = {A.shape[1]} columns")

In [ ]:
# Visualize: the range is a line, the nullspace is a line
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Range (column space)
ax = axes[0]
t_vals = np.linspace(-2, 2, 100)
col = A[:, 0]  # span of this vector
ax.plot(t_vals * col[0], t_vals * col[1], 'b-', linewidth=2, label='Range (column space)')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_title('Range of A')
ax.legend()
ax.grid(True, alpha=0.3)

# Nullspace
ax = axes[1]
null_vec = np.array([-2, 1])
ax.plot(t_vals * null_vec[0], t_vals * null_vec[1], 'r-', linewidth=2, label='Nullspace')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_title('Nullspace of A')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Outer Product and Low-Rank Approximation

Any matrix can be written as a sum of rank-1 outer products. The **SVD** picks the best ones.

In [ ]:
u = np.array([2, 3], dtype=float)
v = np.array([4, 5, 6], dtype=float)

outer = np.outer(u, v)
print("u =", u)
print("v =", v)
print("\nOuter product u v^T =")
print(outer)
print("\nRank =", np.linalg.matrix_rank(outer))

In [ ]:
# Demonstrate low-rank approximation via SVD on a random matrix
np.random.seed(42)
M = np.random.randn(50, 50)

U, S, Vt = np.linalg.svd(M)

# Reconstruct using only the top-k singular values
ranks = [1, 5, 10, 25, 50]
errors = []
for k in ranks:
    M_k = (U[:, :k] * S[:k]) @ Vt[:k, :]
    err = np.linalg.norm(M - M_k) / np.linalg.norm(M)
    errors.append(err)
    print(f"Rank-{k:2d} approximation: relative error = {err:.6f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.semilogy(ranks, errors, 'o-', linewidth=2, markersize=8)
plt.xlabel("Rank of approximation")
plt.ylabel("Relative error")
plt.title("Low-rank approximation via SVD")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Singular values decay — the SVD picks rank-1 pieces in order of importance
plt.figure(figsize=(6, 4))
plt.semilogy(range(1, len(S) + 1), S, 'o-', markersize=4)
plt.xlabel("Index")
plt.ylabel("Singular value")
plt.title("Singular values of a random 50x50 matrix")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()